# AES-256-GCM Decryption

`decrypt_and_verify` recomputes the **tag** and compares it *before*
releasing any plaintext. If even one bit of the ciphertext, tag or aad
was changed, it raises a `ValueError` instead of returning garbage.

In [ ]:
from Crypto.Cipher import AES


def decrypt(key: bytes, nonce: bytes, ciphertext: bytes, tag: bytes,
            aad: bytes = b"") -> bytes:
    """Decrypt AES-GCM data; raises ValueError if anything was tampered with."""
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    cipher.update(aad)
    return cipher.decrypt_and_verify(ciphertext, tag)

## Round trip

Encrypt a message and decrypt it again — you get the original text back.

In [ ]:
from encrypt_aes import encrypt
from generate_aes import generate_key

key = generate_key()
nonce, ciphertext, tag = encrypt(key, b"Hello, AES!")
print("Ciphertext (hex):", ciphertext.hex())
print("Decrypted text  :", decrypt(key, nonce, ciphertext, tag).decode())

## Tampering test

Flip one byte of the ciphertext — GCM must refuse to decrypt it.

In [ ]:
bad_ciphertext = bytearray(ciphertext)
bad_ciphertext[0] ^= 0xFF  # flip every bit of the first byte

try:
    decrypt(key, nonce, bytes(bad_ciphertext), tag)
except ValueError as err:
    print("Tampering detected! Decryption failed:", err)